# Applicant name harmonisation – general pipeline (gold standard)

**Strategy.** 
1. **Clean** each name (accents, legal forms such as INC/LTD/GMBH, common words such as UNIVERSITY → UNIV).
2. **Brand key**: take the first word that is not a generic word (institutions, places, legal forms). Names with the same key form a group.
3. **Fuzzy**: merge keys that are almost identical 

Each group gets an id (`group_id`) and a name (`group_name`). We test the method on the course gold standard with pairwise precision, recall and F1. The functions used are the same provided during lessons 

In [7]:
import re
import unicodedata
from math import comb
import pandas as pd
import networkx as nx
from rapidfuzz.fuzz import ratio

INPUT_CSV = "../PATSTAT_500_label_datav4.csv"
OUTPUT_CSV = "gold_harmonised.csv"
FUZZY_MIN = 90      # min similarity (0-100) to merge two keys
MIN_KEY_LEN = 6     # shorter keys are never fuzzy-merged

## 1. Cleaning

In [8]:
FORMS = [
    "INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION",
    "CO", "COMPANY", "PLC", "GMBH", "MBH", "AG", "KG", "SE", "SA", "SAS", "SARL", "SRL",
    "SPA", "SL", "BV", "NV", "OY", "OYJ", "AB", "AS", "ASA", "APS", "PTY", "PTE", "KK",
    "KABUSHIKI KAISHA", "SP Z OO", "OOO", "BHD", "SDN", "PVT", "LTDA", "LIMITEE", "LTEE",
    "AKTIENGESELLSCHAFT", "SOCIETE ANONYME", "SOCIEDAD ANONIMA", "SOCIETA PER AZIONI"
]
FORMS_RE = "|".join(sorted(FORMS, key=len, reverse=True))
FORMS_CJK = "股份有限公司|有限责任公司|有限責任公司|有限公司|株式会社|有限会社|合同会社|주식회사|公司"
WORDS = {
    "UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITAT": "UNIV", "UNIVERSITE": "UNIV",
    "UNIVERSIDAD": "UNIV", "UNIVERSITA": "UNIV", "INSTITUTE": "INST", "INSTITUT": "INST",
    "INSTITUTO": "INST", "TECHNOLOGY": "TECH", "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH",
    "LABORATORY": "LAB", "LABORATORIES": "LAB", "LABS": "LAB", "CENTRE": "CENTER",
    "SCIENCES": "SCIENCE", "大學": "大学"
}
FILLER = {"OF", "THE", "FOR", "AND", "DE", "DI", "DU", "DES", "ET", "UND"}


def clean(name):
    if pd.isna(name):
        return ""
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", str(name)).upper())
    s = unicodedata.normalize("NFC", "".join(c for i, c in enumerate(s) if not (unicodedata.combining(c) and s[i - 1] < "ɐ")))
    s = re.sub(r"[▲▼?.'’]", "", s)
    latin = re.search(r"[^\x00-\x7f]([A-Z0-9][A-Z0-9 ,&\-/]+)$", s)   # keep the Latin part of CJK + Latin names
    if latin and len(re.sub(r"[^A-Z]", "", latin.group(1))) >= 3:
        s = latin.group(1)
    s = re.sub(r"^C/O\s+", "", s).strip()
    s = re.sub(r"(?:,|\s)\s*\bC/O\b.*$", "", s)
    s = re.sub(r"\(\s*A\s+[A-Z ]+\b(?:CORP|CORPORATION|COMPANY|LIMITED|LTD|INC)\b[^)]*\)", " ", s)
    s = s.replace("A/S", "AS").replace("&", " AND ")
    s = re.sub(rf"^({FORMS_CJK})|({FORMS_CJK})$", "", re.sub(r"[^\w ]|_", " ", s).strip())
    s = " ".join(WORDS.get(w, w) for w in s.split() if w not in FILLER) or s
    s = re.sub(rf"^(({FORMS_RE}) )+|( ({FORMS_RE}))+$", "", s).strip() or s
    return s

## 2. Generic words (skipped when looking for the brand)

In [9]:
INSTITUTIONS = set("""
UNIV INST INSTITUTE COLLEGE ACADEMY SCHOOL HOCHSCHULE ECOLE POLITECNICA POLYTECHNICAL POLYTECHNIC
UNIVERSITETT UNIVERSITAET UNIVERSITAT UNIVERSITE UNIVERSIDAD UNIVERSITA UNIVERSITET TECHNISCHE VOCATIONAL
TECH SCIENCE RESEARCH LAB CENTER GROUP CORP COMPANY COLTD HOLDINGS INTERNATIONAL NATIONAL STATE PUBLIC
ROYAL NORMAL GENERAL CENTRAL FEDERAL LIMITED RESPONSIBILITY KORPERSCHAFT OFFENTLICHEN RECHTS BOARD
REGENTS TRUSTEES FOUNDATION ASSOCIATION PRESIDENT FELLOWS HOSPITAL DEPARTMENT MINISTRY GOVERNMENT AGENCY
SECRETARY DIRECTOR COMMITTEE COUNCIL INSTITUTION ADMINISTRATIVE INDEPENDENT DEGLI STUDI NORMALE SUPERIEURE
POSTS COOPERATION ACADEMIC ACADEMIA FEDERALNOE GOSUDARSTVENNOE BYUDZHETNOE OBRAZOVATELNOE UCHREZHDENIE
VYSSHEGO OBRAZOVANIYA GESELLSCHAFT FORDERUNG ANGEWANDTEN FORSCHUNG WISSENSCHAFTEN EV
""".split())

PLACES = set("""
CHINA CHINESE JAPAN KOREA EUROPE EUROPEAN AMERICA AMERICAN UNITED STATES KINGDOM UK US DEUTSCHLAND
GERMANY FRANCE AUSTRALIA BEIJING SHANGHAI SHENZHEN GUANGDONG ANHUI JIANGSU ZHEJIANG SICHUAN HUBEI HUNAN
HENAN HEBEI FUJIAN SHANDONG SHANXI SHAANXI LIAONING JILIN HEILONGJIANG YUNNAN GUIZHOU GANSU QINGHAI
HAINAN NINGXIA XINJIANG XIZANG GUANGXI INNER MONGOLIA HEFEI NANJING SUZHOU HANGZHOU WUHAN CHENGDU
CHONGQING DALIAN HARBIN TIANJIN XIAN CHANGCHUN SHENYANG ZHENGZHOU CHANGSHA FUZHOU XIAMEN JINAN KUNMING
TAIYUAN DONGGUAN QINGDAO WUXI NORTH SOUTH EAST WEST NORTHEAST NORTHWEST SOUTHEAST SOUTHWEST NEW
COLORADO FLORIDA ARIZONA MICHIGAN OHIO PENN TEXAS CALIFORNIA YORK WASHINGTON VIRGINIA
""".split())

FORMS_WORDS = {w for f in FORMS for w in f.split()}
GENERIC = INSTITUTIONS | PLACES | FORMS_WORDS

## 3. Pipeline: clean → brand key → fuzzy

In [10]:
def brand_key(name, generic):
    "First non-generic word; a one-letter word is joined to the next (D WAVE); whole name if all words are generic."
    words = name.split()
    for i, w in enumerate(words):
        if len(w) == 1 and i + 1 < len(words):
            return w + " " + words[i + 1]
        if w not in generic and len(w) >= 2 and not w.isdigit():
            return w
    return name


def fuzzy_merge(keys):
    "Map each key to the most frequent key among the almost identical ones."
    counts = keys.value_counts()
    long_keys = sorted(k for k in counts.index if len(k) >= MIN_KEY_LEN)
    g = nx.Graph()
    g.add_nodes_from(counts.index)
    g.add_edges_from((a, b) for i, a in enumerate(long_keys) for b in long_keys[i + 1:] if ratio(a, b) >= FUZZY_MIN)
    return {k: min(c, key=lambda x: (-counts[x], x)) for c in nx.connected_components(g) for k in c}


def harmonise(names, generic):
    keys = names.map(clean).map(lambda c: brand_key(c, generic))
    return keys.map(fuzzy_merge(keys))


def make_ids(labels):
    order = labels.value_counts().sort_index().sort_values(ascending=False, kind="stable").index
    return labels.map({lab: f"HARM_{i:05d}" for i, lab in enumerate(order, 1)})


def make_names(names, ids):
    "Most frequent cleaned name of each group (ties: shortest)."
    d = pd.DataFrame({"id": ids, "name": names}).value_counts().reset_index(name="n")
    d["len"] = d["name"].str.len()
    best = d.sort_values(["id", "n", "len", "name"], ascending=[True, False, True, True]).drop_duplicates("id")
    return ids.map(best.set_index("id")["name"])

## 4. Run on the gold standard

In [11]:
gold = pd.read_csv(INPUT_CSV)
gold["group_id"] = make_ids(harmonise(gold["comp"], GENERIC))
gold["group_name"] = make_names(gold["comp"], gold["group_id"])
print(f"{len(gold):,} names -> {gold['group_id'].nunique():,} groups (gold: {gold['Group'].nunique():,})")

3,192 names -> 530 groups (gold: 481)


## 5. Evaluation (pairwise precision, recall, F1)

In [12]:
def pairs_inside(group_sizes):
    "Number of pairs of names inside groups of the given sizes: n(n-1)/2 for each group."
    return sum(comb(int(n), 2) for n in group_sizes)


def precision_recall_f1(gold_groups, predicted_groups):
    "Pairwise precision, recall and F1 of a predicted grouping against the gold groups."
    df = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(df.groupby(["gold", "pred"]).size())         # merged by you and in the gold standard
    tp_fp = pairs_inside(df.groupby("pred").size())                # all the pairs you merged
    tp_fn = pairs_inside(df.groupby("gold").size())                # all the pairs that belong together
    precision = tp / tp_fp if tp_fp else 1.0
    recall = tp / tp_fn
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"groups found": df["pred"].nunique(), "precision": precision, "recall": recall, "F1": f1}


gold[["Group", "comp", "group_id", "group_name"]].to_csv(OUTPUT_CSV, index=False)
pd.Series(precision_recall_f1(gold["Group"], gold["group_id"])).round(3)

groups found    530.000
precision         0.960
recall            0.938
F1                0.949
dtype: float64